# 01. Завантаження, очищення та оптимізація даних (Steam Reviews)

**Мета ноутбука:**
1. Завантажити повний набір даних (`6 417 106` рядків) засобами багатопотокової бібліотеки `Polars`.
2. Заповнити пропуски у назві гри (`app_name`) за допомогою довідника `app_id -> app_name`.
3. Видалити порожні текстові відгуки (`review_text`) та повні дублікати рядків.
4. Виконати оптимізацію типів даних (`downcasting`: `Int64` -> `Int32`/`Int8`, `Categorical`) для зменшення споживання оперативної пам'яті.
5. Згенерувати базові числові ознаки довжини відгуку (`char_len`, `word_count`) та привести цільову змінну `review_score` до бінарного формату `target` (`{0, 1}`).
6. Зафіксувати час кожного етапу та зберегти очищений датасет у колоночний формат **Apache Parquet (zstd)**.

In [1]:
import os
import time
import polars as pl

# Шляхи до вхідного та вихідного файлів
RAW_CSV_PATH = "../data/dataset.csv"
PARQUET_PATH = "../data/steam_reviews_clean.parquet"
os.makedirs("../data", exist_ok=True)

timings = {}

### Крок 1. Читання сирого CSV-файлу та первинна діагностика

In [2]:
t0 = time.perf_counter()
df = pl.read_csv(RAW_CSV_PATH, ignore_errors=True)
timings["1. Читання сирого CSV"] = round(time.perf_counter() - t0, 3)

raw_rows = df.height
raw_mem_mb = round(df.estimated_size("mb"), 2)

print(f"Час завантаження CSV: {timings['1. Читання сирого CSV']} с")
print(f"Розмірність сирого набору: {df.shape}")
print(f"Оцінка споживання RAM: {raw_mem_mb} МБ\n")
print("Кількість пропусків по стовпцях:")
print(df.null_count())

Час завантаження CSV: 1.678 с
Розмірність сирого набору: (6417106, 5)
Оцінка споживання RAM: 2117.17 МБ

Кількість пропусків по стовпцях:
shape: (1, 5)
┌────────┬──────────┬─────────────┬──────────────┬──────────────┐
│ app_id ┆ app_name ┆ review_text ┆ review_score ┆ review_votes │
│ ---    ┆ ---      ┆ ---         ┆ ---          ┆ ---          │
│ u32    ┆ u32      ┆ u32         ┆ u32          ┆ u32          │
╞════════╪══════════╪═════════════╪══════════════╪══════════════╡
│ 0      ┆ 183234   ┆ 7305        ┆ 0            ┆ 0            │
└────────┴──────────┴─────────────┴──────────────┴──────────────┘


### Крок 2. Заповнення пропусків у `app_name`, видалення пустих відгуків та дублікатів

In [3]:
t0 = time.perf_counter()

# Будуємо довідник відповідності app_id -> app_name з непорожніх записів
app_mapping = (
    df.filter(pl.col("app_name").is_not_null())
    .group_by("app_id")
    .agg(pl.col("app_name").first().alias("mapped_name"))
)

# Заповнюємо пропуски, видаляємо рядки без тексту відгуку та прибираємо повні дублікати
df_clean = (
    df.join(app_mapping, on="app_id", how="left")
    .with_columns(
        pl.coalesce(["app_name", "mapped_name"]).fill_null("Unknown Game").alias("app_name")
    )
    .drop("mapped_name")
    .filter(
        pl.col("review_text").is_not_null() &
        (pl.col("review_text").str.strip_chars().str.len_chars() > 0)
    )
    .unique()
)

del df  # звільняємо оперативну пам'ять
timings["2. Очищення пропусків та дублікатів"] = round(time.perf_counter() - t0, 3)
print(f"Час очищення: {timings['2. Очищення пропусків та дублікатів']} с")
print(f"Залишилось унікальних записів: {df_clean.height:,}")

Час очищення: 4.407 с
Залишилось унікальних записів: 4,618,780


### Крок 3. Оптимізація типів даних (Downcasting) та базові текстові ознаки

In [4]:
t0 = time.perf_counter()

df_clean = df_clean.with_columns([
    pl.col("app_id").cast(pl.Int32),
    pl.col("app_name").cast(pl.Categorical),
    pl.when(pl.col("review_score") == 1).then(1).otherwise(0).cast(pl.Int8).alias("target"),
    pl.col("review_votes").clip(lower_bound=0).cast(pl.Int32),
    pl.col("review_text").str.len_chars().cast(pl.Int32).alias("char_len"),
    pl.col("review_text").str.split(" ").list.len().cast(pl.Int32).alias("word_count"),
]).drop("review_score")

timings["3. Downcasting та генерація ознак"] = round(time.perf_counter() - t0, 3)
clean_mem_mb = round(df_clean.estimated_size("mb"), 2)
print(f"Час оптимізації типів: {timings['3. Downcasting та генерація ознак']} с")
print(f"Споживання RAM після оптимізації: {clean_mem_mb} МБ (було {raw_mem_mb} МБ)")

Час оптимізації типів: 10.807 с
Споживання RAM після оптимізації: 1723.79 МБ (було 2117.17 МБ)


### Крок 4. Збереження у формат Apache Parquet (zstd) та підсумкова таблиця часу

In [5]:
t0 = time.perf_counter()
df_clean.write_parquet(PARQUET_PATH, compression="zstd")
timings["4. Запис у Parquet (zstd)"] = round(time.perf_counter() - t0, 3)

parquet_size_mb = round(os.path.getsize(PARQUET_PATH) / (1024 * 1024), 2)

print("=" * 55)
print(f"ПІДСУМОК: {raw_rows:,} -> {df_clean.height:,} рядків ({df_clean.width} стовпців)")
print(f"Розмір Parquet на диску: {parquet_size_mb} МБ")
print("=" * 55)
for step, sec in timings.items():
    print(f"{step:<38} : {sec:>6.3f} с")

ПІДСУМОК: 6,417,106 -> 4,618,780 рядків (7 стовпців)
Розмір Parquet на диску: 666.65 МБ
1. Читання сирого CSV                  :  1.678 с
2. Очищення пропусків та дублікатів    :  4.407 с
3. Downcasting та генерація ознак      : 10.807 с
4. Запис у Parquet (zstd)              :  2.380 с
